# hive-bot supervised pretraining (real human games)

Runs `pretrain()` (see `src/hive_bot/training/pretrain.py`) over real games scraped from hivegame.com -- behavior-cloning the network on human moves and game outcomes, as a warm start before self-play (`train_colab.ipynb`) takes over. See the plan doc, "Bootstrap training from real hivegame.com games".

This shares `CHECKPOINT_DIR` with `train_colab.ipynb` and uses the exact same "resume from the latest checkpoint in Drive" pattern, so the two notebooks compose freely in either order/repeatedly: pretrain -> self-play -> more human data -> pretrain again -> ..., each picking up wherever the other left off.

Runtime: GPU optional, same as `train_colab.ipynb` -- helps the network forward/backward passes, not required.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

## 1. Install hive-bot

Same as `train_colab.ipynb` -- pick ONE of the two cells below.

In [ ]:
# Option A: install from a git remote.

REPO_URL = "https://github.com/JODLYO/hive-bot.git"

!git clone {REPO_URL} /content/hive-bot 2>/dev/null || (cd /content/hive-bot && git pull)

!pip install -q /content/hive-bot

In [ ]:
# Option B: install a local copy already sitting on Google Drive (mount

# Drive first -- see the next section -- then point this at the repo root).

# LOCAL_REPO_PATH = "/content/drive/MyDrive/hive-bot"

# !pip install -q {LOCAL_REPO_PATH}

## 2. Mount Google Drive

`CHECKPOINT_DIR` is the **same path** `train_colab.ipynb` uses, so whichever notebook ran most recently is what the other resumes from.

`TRAIN_DATA_PATH`/`VAL_DATA_PATH` are the train/val JSONL splits built by `scripts/build_pretrain_dataset.py` -- run that locally (`make scrape-games && make build-pretrain-dataset`; `data/` isn't committed to git, so this can't just be pulled from the repo clone above) and upload the resulting `data/hivegame_samples/train_games.jsonl` and `data/hivegame_samples/val_games.jsonl` to these paths on Drive before running the cells below. These are just the raw game records (history + result), not pre-encoded tensors -- `pretrain()` replays/encodes a bounded number of games at a time itself (see its module docstring), since materializing the full archive as encoded samples would need on the order of hundreds of GB. The split happens on whole *games* (10% held out, see the script's docstring), not on individual positions, so val loss reflects generalization to genuinely unseen games rather than unseen positions within games the model has already partly trained on. Re-scrape and re-upload whenever you want to pretrain on a fresher/larger batch of games.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = Path("/content/drive/MyDrive/hive-bot-checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/content/drive/MyDrive/hive-bot-pretrain-data")
TRAIN_DATA_PATH = DATA_DIR / "train_games.jsonl"
VAL_DATA_PATH = DATA_DIR / "val_games.jsonl"

## 3. Load the human-game dataset

In [ ]:
from hive_bot.data.hivegame_archive import load_base_games_jsonl

games = load_base_games_jsonl(TRAIN_DATA_PATH)
val_games = load_base_games_jsonl(VAL_DATA_PATH)

print(f"{len(games)} train games loaded from {TRAIN_DATA_PATH}")
print(f"{len(val_games)} val games loaded from {VAL_DATA_PATH}")

## 4. Configuration

In [ ]:
from hive_bot.engine.constants import BASE_PIECE_TYPES

ENABLED_TYPES = BASE_PIECE_TYPES  # matches the scraped data -- expansions=false, see scrape_hivegame_archive.py

EPOCHS = 5
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
SEED = 0

# How many games' worth of samples to replay/encode into memory at
# once (see pretrain.py's module docstring) -- Colab typically has
# more RAM than a laptop, so this can go higher than the library's
# own default (200) if you want faster epochs and have the memory
# to spare.
GAMES_PER_CHUNK = 200

# HiveNet(**NETWORK_KWARGS) -- {} uses the real default size (64 channels,
# 6 residual blocks), matching train_colab.ipynb's default so a
# pretrained checkpoint's architecture always matches what self-play
# expects to resume into.
NETWORK_KWARGS: dict = {}

## 5. Resume (or start) the model

Finds the highest-numbered checkpoint already in `CHECKPOINT_DIR` -- from a previous self-play run, a previous pretrain run, or both -- and loads its weights as pretraining's starting point; starts a fresh network if there isn't one yet. Identical helper to `train_colab.ipynb`'s.

In [ ]:
import re

from hive_bot.model.network import HiveNet


def latest_checkpoint(checkpoint_dir: Path) -> Path | None:

    checkpoints = list(checkpoint_dir.glob("checkpoint_*.pt"))

    if not checkpoints:
        return None

    return max(
        checkpoints, key=lambda p: int(re.search(r"checkpoint_(\d+)\.pt", p.name).group(1))
    )


resume_from = latest_checkpoint(CHECKPOINT_DIR)

model = HiveNet(**NETWORK_KWARGS)

if resume_from is not None:
    checkpoint = torch.load(resume_from, map_location="cpu")

    model.load_state_dict(checkpoint["model"])

    print("starting from:", resume_from)

else:
    print("starting from: a fresh, untrained network")

## 6. Pretrain

Re-run this cell as many times as you like -- each run continues from whatever `model` currently holds (either the loaded checkpoint from the cell above, or the result of the last time this cell ran).

In [ ]:
from hive_bot.training.pretrain import pretrain

model = pretrain(
    model,
    games,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    games_per_chunk=GAMES_PER_CHUNK,
    lr=LEARNING_RATE,
    val_games=val_games,
    seed=SEED,
)

## 7. Hand off to self-play

Saves the pretrained model into `CHECKPOINT_DIR` using `train_colab.ipynb`'s own `checkpoint_<iteration>.pt` naming -- so its `latest_checkpoint()` picks this up automatically next time that notebook runs, continuing self-play from here with **no changes needed there**. Iteration number continues from whatever `resume_from` was (self-play or a previous pretrain round), or starts at 0 if this was the very first checkpoint.

In [ ]:
iteration = 0

if resume_from is not None:
    iteration = int(re.search(r"checkpoint_(\d+)\.pt", resume_from.name).group(1)) + 1

torch.save(
    {
        "model": model.state_dict(),
        # pretrain() uses its own fresh optimizer each run rather than
        # preserving self-play's Adam momentum across the switch --
        # train_colab.ipynb's resume_from only *needs* an
        # "optimizer" key to exist, its contents don't have to relate
        # to the model's pretraining history.
        "optimizer": torch.optim.Adam(model.parameters()).state_dict(),
        "iteration": iteration,
    },
    CHECKPOINT_DIR / f"checkpoint_{iteration}.pt",
)

print(
    f"saved checkpoint_{iteration}.pt -- train_colab.ipynb will resume self-play from here"
)

## 8. Try the current model

Same sanity check as `train_colab.ipynb`'s last cell.

In [ ]:
from hive_bot.analysis.bot import HiveBot
from hive_bot.engine.state import GameState

bot = HiveBot(model, num_simulations=800)

state = GameState.new_game(ENABLED_TYPES)

analysis = bot.analyze(state)

print(f"win probability for the player to move: {analysis.win_probability:.1%}")

print("top moves:")

for evaluation in analysis.move_evaluations[:5]:
    print(f"  {evaluation.move}  (visited {evaluation.visit_fraction:.1%} of the time)")